## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


# 1 Agregado de productos no vendidos

In [ ]:
import os as os
import duckdb

In [ ]:
# defino los parametros
PARAM = {'experimento':'WF-601',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [ ]:
# creo la carpeta del experimento
ruta = "/content/buckets/b1/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1.1 Creacion de tablas
Utilizo DuckDB
<br>Cargo los archivos a utilizar en TABLAS de DuckDB

In [ ]:
con = duckdb.connect()

In [ ]:
# creo la tabla del sell-in

con.execute("""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT customer_id,
           product_id,
           periodo,
           plan_precios_cuidados,
           cust_request_qty,
           cust_request_tn,
           tn
    FROM read_csv_auto('/content/datasets/sell-in.txt.gz')
    ORDER BY customer_id, product_id, periodo
""")

In [ ]:
# creo tabla de periodos

con.execute("""
    CREATE OR REPLACE TABLE tb_periodos AS
    SELECT DISTINCT periodo
    FROM   tb_sellin
    ORDER BY 1
""")

In [ ]:
# la fecha de nacimiento y muerte de los productos

con.execute("""
    CREATE OR REPLACE TABLE tb_productos_fechas AS
    SELECT product_id,
           MIN(periodo) as periodo_min,
           MAX(periodo) as periodo_max
    FROM   tb_sellin
    GROUP BY product_id
""")

In [ ]:
# la fecha de primera compra de cada cliente

con.execute("""
    CREATE OR REPLACE TABLE tb_clientes_fechas AS
    SELECT customer_id,
           MIN(periodo) as periodo_min
    FROM   tb_sellin
    GROUP BY customer_id
""")

In [ ]:
# la tabla con los precios cuidados

con.execute("""
    CREATE OR REPLACE TABLE tb_precios_cuidados AS
    SELECT
           product_id,
           MIN(periodo) AS periodo_min,
           MAX(periodo) AS periodo_max
    FROM tb_sellin
    WHERE plan_precios_cuidados = 1
    GROUP BY  product_id
    ORDER BY 1
""")

## 1.2 Producto Cartesiano de zeros

In [ ]:
# el producto cartesiano
# si ya no existe

con.execute("""
CREATE OR REPLACE TABLE tb_zeroes AS
SELECT cf.customer_id,
       pf.product_id,
       per.periodo,
       CAST(0 AS INT) AS plan_precios_cuidados,
       CAST(0 AS INT) AS cust_request_qty,
       0.0 AS cust_request_tn,
       0.0 AS tn
FROM   tb_productos_fechas pf,
       tb_clientes_fechas cf,
       tb_periodos  per
WHERE
       NOT EXISTS (
         SELECT 1
         FROM   tb_sellin si
         WHERE  si.periodo = per.periodo
         AND    si.customer_id = cf.customer_id
         AND    si.product_id = pf.product_id
       )
AND    per.periodo BETWEEN pf.periodo_min AND pf.periodo_max
AND    per.periodo >= cf.periodo_min
ORDER BY 1, 2, 3
""")

In [ ]:
# actualizo precios cuidados

con.execute("""
UPDATE tb_zeroes  z
SET plan_precios_cuidados = 1
WHERE  EXISTS ( SELECT  1
                FROM tb_precios_cuidados p
                WHERE z.periodo BETWEEN  p.periodo_min AND p.periodo_max
                AND   p.product_id = z.product_id)
""")

In [ ]:
con.sql("""
SELECT  COUNT(*)
FROM    tb_zeroes
""").show()

## 1.3 Tabla Final

In [ ]:
# creo la nueva tabla
# ,  agrupando y sumando/max

con.execute("""
CREATE OR REPLACE TABLE tb_sellin_zeroes AS
SELECT *
FROM   tb_sellin
UNION ALL
SELECT *
FROM   tb_zeroes
ORDER BY 1, 2, 3
""")

In [ ]:
# cuento cuantos registros tiene la nueva tabla
con.sql("""
SELECT  COUNT(*)
FROM    tb_sellin_zeroes
""").show()

In [ ]:
# una vista de la nueva tabla
con.sql("""
SELECT  *
FROM    tb_sellin_zeroes
""").show()

## 1.4  Grabar dataset

In [ ]:
# grabo el nuevo dataset en disco, carpeta datasets

con.execute("""
COPY (SELECT * FROM tb_sellin_zeroes ORDER BY 1,2,3)
TO '/content/buckets/b1/datasets/sell-in-zeroes.txt.gz'
(FORMAT csv, COMPRESSION 'gzip');
""")